# Team 04 — Site Grid & Side Alignment (Phase 3)

Real buildings are **not** dropped at arbitrary rotations inside a plot. They sit on a site grid,
**parallel to a preferred boundary** (the street frontage / the longest edge / the main-road side).
This notebook replaces the free 5 m sweep + 36 free rotations with **grid-node positions × aligned
orientations**, on a **complex non-orthogonal site**.

What this adds over free placement:

1. A **site grid** derived from a *chosen side* — buildings can only take the {parallel,
   perpendicular} orientations of that side (`site_grid.derive_site_grid` / `aligned_orientations`).
2. **It no longer looks random** — free vs. grid-aligned, side by side, for identical fitness.
3. **Obtuse footprints**: an L tucked into a splayed (non-orthogonal) corner lets its free wing
   follow the *adjacent* side, so its arms spread to the corner's interior angle (> 90°).
4. **Use-driven placement**: a **commercial** building hugs the chosen frontage
   (`boundary_proximity`), while residential leans on view + sun.
5. Two or more buildings placed **together**, each aligned and clearing the rest.

Deterministic (no LLM); only matplotlib is needed.

In [ ]:
from __future__ import annotations
import sys, math
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (workspace_root, workspace_root.parent,
                   workspace_root / 'team_04', workspace_root.parent / 'team_04')
TEAM_ROOT = next((p for p in candidate_roots if (p / 'agent').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run from workspace root, team_04, or team_04/test_notebooks.')
if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from agent.tools.site_model import build_site_model
from agent.tools.building_shape_graph import build_shape_model
from agent.tools.parametric_shape import apply_shape_variables, shape_variable_spec
from agent.tools.site_grid import (
    derive_site_grid, aligned_orientations, align_building_to_grid, alignment_score,
    snap_to_grid, corner_interior_angle, corner_wing_rotation,
)
from agent.tools.view_optimizer import (
    sample_valid_placements, optimize_aligned_placement, place_buildings_aligned,
)
print('Team root:', TEAM_ROOT)

## 1. Complex site + the derived grid

A splayed, non-orthogonal pentagon (no right angles). The grid is derived from the **longest side**
by default (the sensible fallback before roads land in Phase 2) — origin + two axes, clipped to the
buildable zone, drawn as grid lines with seed nodes.

In [ ]:
SITE = [[0, 0, 0], [130, 18, 0], [150, 92, 0], [62, 128, 0], [-14, 74, 0], [0, 0, 0]]
site_model = build_site_model(SITE, {'default_setback': 6.0})
grid = derive_site_grid(site_model, spacing=12.0)   # default: longest side

print('chosen side   :', grid['alignment_side_index'], grid['alignment_side_label'])
print('grid angle    :', grid['angle_deg'], 'deg')
print('orientations  :', aligned_orientations(grid), '(parallel, perpendicular)')
print('grid nodes    :', grid['node_count'])

def draw_site(ax, model=site_model, site=SITE):
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in site[:-1]], fc='#f0ede6', ec='#555', lw=2, zorder=1))
    bz = (model.get('setbacks') or {}).get('buildable_boundary')
    if bz:
        ax.add_patch(plt.Polygon([(p[0], p[1]) for p in bz[:-1]], fc='none', ec='#27ae60', lw=1.3, ls='--', zorder=2))
    ax.set_aspect('equal')

def draw_grid(ax, g, nodes=True):
    for ln in g['grid_lines']:
        ax.plot([ln[0][0], ln[1][0]], [ln[0][1], ln[1][1]], color='#b0c4de', lw=0.6, zorder=2)
    if nodes:
        xs = [n[0] for n in g['grid_nodes']]; ys = [n[1] for n in g['grid_nodes']]
        ax.scatter(xs, ys, s=8, color='#5d6d7e', zorder=3)
    # highlight the chosen side
    i = g['alignment_side_index']; coords = [(p[0], p[1]) for p in SITE[:-1]]
    a = coords[i]; b = coords[(i + 1) % len(coords)]
    ax.plot([a[0], b[0]], [a[1], b[1]], color='#e67e22', lw=4, zorder=4, label='chosen side')

fig, ax = plt.subplots(figsize=(8, 7))
draw_site(ax); draw_grid(ax, grid)
ax.legend(loc='upper right'); ax.set_title('Complex site + grid aligned to the chosen (longest) side')
ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## 2. Free placement vs. grid-aligned placement

The same building, the same site. Left: the old free sweep (any of 36 rotations, anywhere). Right:
restricted to grid nodes × {parallel, perpendicular}. The right reads as *intentional* — every
candidate is parallel to the chosen frontage.

In [ ]:
def base_boundary(btype, area, depth=12.0, ratio=0.5):
    poly = build_shape_model(area=area, building_type=btype, building_depth=depth, shape_ratio=ratio).polygon
    return [[round(float(x), 3), round(float(y), 3), 0.0] for x, y in poly.exterior.coords]

bldg = base_boundary('I', 320.0)
SET = {'default_setback': 6.0}

free = sample_valid_placements(bldg, SITE, rotation_step_degrees=10, grid_step=8.0, site_setbacks=SET)
aligned = sample_valid_placements(bldg, SITE, grid=grid, site_setbacks=SET)
print(f'free candidates    : {len(free)}  (mixed rotations)')
print(f'aligned candidates : {len(aligned)}  (all parallel/perpendicular to the chosen side)')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, cands, title, show_grid in ((axes[0], free[::7], 'FREE sweep (rotates anywhere)', False),
                                     (axes[1], aligned[::3], 'GRID-ALIGNED (parallel to chosen side)', True)):
    draw_site(ax)
    if show_grid: draw_grid(ax, grid, nodes=False)
    for c in cands:
        ax.add_patch(plt.Polygon([(p[0], p[1]) for p in c['boundary'][:-1]],
                                 fc='#2980b9', ec='#1b4f72', lw=0.6, alpha=0.25))
    ax.set_title(title); ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## 3. Obtuse L — wings follow two non-orthogonal sides

When an **L** is tucked into a splayed site corner, its main wing aligns to the chosen side and its
free wing follows the *adjacent* side. Because that corner is obtuse, the L's arms spread **> 90°**
instead of a rigid right angle. `corner_wing_rotation` returns exactly the leaf-wing rotation needed.

In [ ]:
side_i = grid['alignment_side_index']
corner = (side_i + 1) % (len(SITE) - 1)
theta = corner_interior_angle(site_model, corner)
wing_rot = corner_wing_rotation(site_model, side_i)
print(f'site corner {corner} interior angle: {theta} deg  ->  leaf-wing rotation: {wing_rot} deg')

# Build a rigid (90 deg) L and an obtuse L (free wing bent to follow the adjacent side).
AREA_L = 700.0
spec = shape_variable_spec('L', AREA_L)
n_leaf = len(spec['leaf_wing_indices'])
rigid_L = build_shape_model(area=AREA_L, building_type='L', building_depth=13.0, shape_ratio=0.5).polygon
rigid_Lb = [[x, y, 0.0] for x, y in rigid_L.exterior.coords]
obtuse_L = apply_shape_variables('L', AREA_L, [13.0, 0.5] + [wing_rot] * n_leaf, spec['leaf_wing_indices'])
obtuse_Lb = [[x, y, 0.0] for x, y in obtuse_L.exterior.coords]

# Drop both at the splayed corner, main wing aligned to the chosen side.
coords = [(p[0], p[1]) for p in SITE[:-1]]
corner_pt = coords[corner]
cx = corner_pt[0] + (sum(p[0] for p in coords) / len(coords) - corner_pt[0]) * 0.28
cy = corner_pt[1] + (sum(p[1] for p in coords) / len(coords) - corner_pt[1]) * 0.28
rigid_placed = align_building_to_grid(rigid_Lb, grid, [cx, cy], aligned_orientations(grid)[0])
obtuse_placed = align_building_to_grid(obtuse_Lb, grid, [cx, cy], aligned_orientations(grid)[0])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, placed, title, c in ((axes[0], rigid_placed, f'Rigid L (90\u00b0 wings)', '#7f8c8d'),
                              (axes[1], obtuse_placed, f'Obtuse L (wings spread {theta:.0f}\u00b0 to follow the site)', '#16a085')):
    draw_site(ax); draw_grid(ax, grid, nodes=False)
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in placed[:-1]], fc=c, ec='k', lw=1.4, alpha=0.7, zorder=5))
    ax.set_title(title); ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## 4. Use-driven placement — commercial hugs the frontage

Same building, same grid, different `use`. **Commercial / office / retail** buildings pick up a
`boundary_proximity` objective and line the chosen frontage; **residential** leans on view + sun and
sits back. The optimizer is exhaustive over the (small) aligned candidate set, so the best option
is exact.

In [ ]:
from agent.tools.view_analysis import _coerce_polygon_2d
site_poly = _coerce_polygon_2d(SITE)
coords = [(p[0], p[1]) for p in SITE[:-1]]
ref_line = [list(coords[side_i]), list(coords[(side_i + 1) % len(coords)])]   # the chosen side segment

shop = base_boundary('I', 360.0, depth=14.0)
results = {}
for use in ('commercial', 'residential'):
    res = optimize_aligned_placement(
        base_boundary=shop, site_boundary=SITE, grid=grid, use=use,
        reference_line=ref_line, site_setbacks=SET, saved_option_count=6,
    )
    best = res['options'][0]
    dist = ref_line_dist = _coerce_polygon_2d(best['boundary']).distance(
        __import__('shapely').geometry.LineString(ref_line))
    results[use] = (res, best, dist)
    print(f"{use:>11}: objectives={[c['name'] for c in res['objective_configs']]}")
    print(f"{'':>11}  best distance to chosen frontage = {dist:6.2f} m   (combined {best['combined_score']:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, use in zip(axes, ('commercial', 'residential')):
    res, best, dist = results[use]
    draw_site(ax); draw_grid(ax, grid, nodes=False)
    ax.plot([ref_line[0][0], ref_line[1][0]], [ref_line[0][1], ref_line[1][1]], color='#e67e22', lw=4, zorder=4)
    for opt in res['options'][1:]:
        ax.add_patch(plt.Polygon([(p[0], p[1]) for p in opt['boundary'][:-1]], fc='none', ec='#aab', lw=0.8, ls='--', zorder=4))
    c = '#c0392b' if use == 'commercial' else '#2980b9'
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in best['boundary'][:-1]], fc=c, ec='k', lw=1.5, alpha=0.75, zorder=6))
    ax.set_title(f'{use} \u2014 best sits {dist:.1f} m from the frontage')
    ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## 5. Two buildings, placed together

A **commercial** block (lines the frontage) and a **residential** block (set back, view + sun),
placed greedily on the grid — each aligned, each clearing the other by ≥ 6 m. Both follow the site
grid; neither is at a random angle.

In [ ]:
from agent.tools.sun_analysis import compute_sun_vectors
sun_vectors = compute_sun_vectors()   # worst-case western sun

layout = place_buildings_aligned(
    [{'base_boundary': base_boundary('I', 360.0, depth=14.0), 'use': 'commercial'},
     {'base_boundary': base_boundary('L', 620.0, depth=12.0), 'use': 'residential'}],
    SITE, grid, site_setbacks=SET, reference_line=ref_line,
    sun_vectors=sun_vectors, sun_weight=0.5, min_separation=6.0,
)
print('placed:', layout['placed_count'])
for b in layout['buildings']:
    print(f"  building {b['building_index']} ({b['use']:>11}): align={b['alignment_score']:.3f}  "
          f"view={b['unblocked_view_score']:.2f}  combined={b['combined_score']:.3f}")

fig, ax = plt.subplots(figsize=(9, 8))
draw_site(ax); draw_grid(ax, grid, nodes=False)
ax.plot([ref_line[0][0], ref_line[1][0]], [ref_line[0][1], ref_line[1][1]], color='#e67e22', lw=4, zorder=4, label='chosen frontage')
for b, c in zip(layout['buildings'], ('#c0392b', '#2980b9')):
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in b['boundary'][:-1]], fc=c, ec='k', lw=1.5, alpha=0.75, zorder=6))
    cen = b['centroid_xy']; ax.text(cen[0], cen[1], b['use'], ha='center', fontsize=8, weight='bold', zorder=7)
ax.legend(loc='upper right'); ax.set_title('Two aligned buildings \u2014 commercial on the frontage, residential set back')
ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## Summary

- `site_grid.derive_site_grid` builds a placement grid from a **chosen site side** on an arbitrary
  (non-orthogonal) site; `aligned_orientations` is the only orientation set a building may take —
  **no free rotation**.
- `optimize_aligned_placement` exhaustively ranks grid-node × aligned-orientation candidates with a
  **use-driven** objective mix (commercial → `boundary_proximity`, residential → view + sun), and
  `place_buildings_aligned` sequences several buildings, each aligned and clearing the rest.
- `corner_wing_rotation` lets an L follow a splayed corner so its wings spread **obtuse**, matching
  the real site geometry.

Backend: `agent/tools/site_grid.py` + `grid_alignment`/`boundary_proximity` objectives and
`optimize_aligned_placement`/`place_buildings_aligned`/grid-aware `sample_valid_placements` in
`view_optimizer.py`. Regressions: `benchmarking/test_site_grid.py` (15 tests). Frontend overlay:
`frontend/site/GridOverlay.tsx` via `POST /tools/{site_grid,aligned_placement}`.